# Sandbox Compute — Quickstart

**Sandbox Compute** lets you run supported open-weight models (Gemma, Qwen, Whisper, OmniVoice, FLUX, RT-DETR, …) on dedicated, warm GPU compute inside your VideoDB workflows. You create a *sandbox*, wait until it's active, then pass its `sandbox_id` to any supported API — VideoDB routes that job to your sandbox instead of the default hosted path.

This notebook walks through the full surface end to end:

| Step | What it shows |
|---|---|
| **Setup** | Install the SDK, connect to dev |
| **1. Lifecycle** | Create, wait, retrieve, list a sandbox |
| **2. Media** | Upload one sample video reused by the understanding steps |
| **3. Text** | `generate_text` — Qwen |
| **4. Visual understanding** | `video.understand` (VLM) — Qwen |
| **5. Object detection** | `video.understand` — RT-DETR |
| **6. Text-to-speech** | `generate_voice` — OmniVoice |
| **7. Image generation** | `generate_image` — FLUX |
| **Cleanup** | Stop the sandbox |

### Key concepts
- **Tier** — sandbox size (`small` / `medium`). Pick the smallest tier that supports your **largest** model. FLUX needs `medium`, so this notebook uses `medium`.
- **`sandbox_id` routing** — pass it to select your warm sandbox; omit it to use VideoDB's default hosted models.
- **Sync vs async** — text & understanding return results directly; generation (voice/image) returns a **job** you `wait()` on.

> 💸 **A running sandbox consumes compute.** Keep it active while you run the workload sections, then run the final **Cleanup** cell to release it.

## Setup

### Install the SDK
Install the VideoDB SDK (**`videodb==0.5.2`**), which includes the Sandbox interface.

In [10]:
!pip install -q videodb==0.5.2 python-dotenv


[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


### Connect to VideoDB dev
Uses a dev API key entered securely and connects explicitly to the dev endpoint.

In [ ]:
from getpass import getpass

from dotenv import load_dotenv
from videodb import connect

load_dotenv()

api_key = getpass("Enter your VideoDB API key: ")
if not api_key:
    raise ValueError("A VideoDB API key is required.")

conn = connect(api_key=api_key)
collection = conn.get_collection()
print("Connected to VideoDB")

Connected to VideoDB


## 1. Sandbox lifecycle

A **sandbox** is a dedicated pool of warm GPU compute holding one or more models. You create it once and reuse it across every compatible job below.

### Create a sandbox
We request all four models this notebook uses. FLUX requires the `medium` tier, and `medium` also covers the others, so we create a single `medium` sandbox.

> The SDK's `SandboxTier` is imported from `videodb`; model names are plain strings (see the [model catalog](https://docs.videodb.io) for the full list + minimum tiers).

In [12]:
from datetime import datetime, timezone

from videodb import SandboxTier

# Model IDs (plain strings — see the sandbox model catalog).
TEXT_AND_VISION_MODEL = "Qwen/Qwen3.5-9B"          # text + vision (VLM)
OBJECT_DETECTION_MODEL = "rtdetr-v2-r50vd"          # object detection
TTS_MODEL = "k2-fsa/OmniVoice"                      # text-to-speech
IMAGE_GENERATION_MODEL = "black-forest-labs/FLUX.1-dev"  # image generation

SANDBOX_TIER = SandboxTier.medium
SANDBOX_MODELS = [
    TEXT_AND_VISION_MODEL,
    OBJECT_DETECTION_MODEL,
    TTS_MODEL,
    IMAGE_GENERATION_MODEL,
]
SANDBOX_NAME = f"sandbox-cookbook-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}"

sandbox = conn.create_sandbox(
    tier=SANDBOX_TIER,
    name=SANDBOX_NAME,
    models=SANDBOX_MODELS,
)

print("Sandbox ID:", sandbox.id)
print("Status:", sandbox.status)
print("Tier:", sandbox.tier)
print("Models:", sandbox.models)

Sandbox ID: bx-13832c0a9e4f4fa9
Status: provisioning
Tier: medium
Models: ['Qwen/Qwen3.5-9B', 'rtdetr-v2-r50vd', 'k2-fsa/OmniVoice', 'black-forest-labs/FLUX.1-dev']


### Wait until the sandbox is active
Creation returns immediately while compute provisions in the background. Submit jobs only after `wait_for_ready()` returns.

The SDK treats both `active` and `alert` as ready. `alert` can mean only *part* of a multi-model sandbox came up — so the per-workload cells below remain the authoritative check for each model.

In [13]:
sandbox.wait_for_ready(timeout=1200, interval=5)

print("Sandbox ID:", sandbox.id)
print("Status:", sandbox.status)
print("Ready:", sandbox.is_ready)
print("Active:", sandbox.is_active)

Sandbox ID: bx-13832c0a9e4f4fa9
Status: active
Ready: True
Active: True


### Retrieve and list sandboxes
`get_sandbox(id)` fetches a sandbox by ID; `refresh()` pulls the latest server state; `list_sandboxes()` enumerates them (optionally filtered by status).

In [14]:
# Retrieve a single sandbox by ID and refresh its state
same_sandbox = conn.get_sandbox(sandbox.id)
same_sandbox.refresh()
print("Retrieved:", {
    "id": same_sandbox.id,
    "name": same_sandbox.name,
    "tier": same_sandbox.tier,
    "status": same_sandbox.status,
    "models": same_sandbox.models,
})

# List all / active sandboxes
all_sandboxes = conn.list_sandboxes()
active_sandboxes = conn.list_sandboxes(status="active")
print("All sandboxes:", len(all_sandboxes))
print("Active sandbox IDs:", [item.id for item in active_sandboxes])

Retrieved: {'id': 'bx-13832c0a9e4f4fa9', 'name': 'sandbox-cookbook-20260725-042808', 'tier': 'medium', 'status': 'active', 'models': ['Qwen/Qwen3.5-9B', 'rtdetr-v2-r50vd', 'k2-fsa/OmniVoice', 'black-forest-labs/FLUX.1-dev']}
All sandboxes: 20
Active sandbox IDs: ['bx-13832c0a9e4f4fa9', 'bx-015fd8939f184276']


## 2. Prepare shared test media

The visual-understanding and object-detection steps reuse one short public video. `upload()` waits for ingestion and returns a `Video`.

In [15]:
SAMPLE_VIDEO_URL = "https://www.youtube.com/watch?v=jNQXAC9IVRw"

video = collection.upload(
    url=SAMPLE_VIDEO_URL,
    name="Sandbox cookbook sample",
)

print("Collection ID:", collection.id)
print("Video ID:", video.id)
print("Video length:", video.length)

Collection ID: c-7a9dacb6-2ff3-481a-aef7-bc747ba89340
Video ID: m-z-019f9796-8ff5-7963-b2ba-fe344c310f7c
Video length: 18.0


## 3. Text generation — Qwen

The smallest end-to-end check: pass `model_name` + `sandbox_id` and the server validates the sandbox, routes the completion to your warm Qwen, and returns text directly (synchronous).

> **Reasoning models:** Qwen3-family models can emit `<think>…</think>` reasoning. If you only see reasoning, raise `max_tokens` (done here) so the final answer isn't truncated.

In [16]:
text_response = collection.generate_text(
    prompt=(
        "In three concise bullet points, explain why dedicated inference "
        "compute is useful for video AI workflows."
    ),
    model_name=TEXT_AND_VISION_MODEL,
    sandbox_id=sandbox.id,
    max_tokens=512,
    temperature=0.2,
)

print("Model:", TEXT_AND_VISION_MODEL)
print("Response:\n", text_response)

Model: Qwen/Qwen3.5-9B
Response:
 {'output': 'Thinking Process:\n\n1.  **Analyze the Request:**\n    *   Topic: Dedicated inference compute for video AI workflows.\n    *   Format: Three concise bullet points.\n    *   Goal: Explain why it is useful.\n\n2.  **Identify Key Benefits of Dedicated Inference Compute for Video AI:**\n    *   Video AI is computationally intensive (high resolution, high frame rates, complex models).\n    *   Latency matters (real-time or near real-time processing).\n    *   Cost/Efficiency (GPU utilization, avoiding shared resource contention).\n    *   Reliability/Consistency (SLA, predictable performance).\n    *   Scalability (handling bursts of traffic).\n\n3.  **Draft Initial Points:**\n    *   Video processing needs a lot of power, so dedicated GPUs help speed it up.\n    *   It reduces latency so you can process video in real-time without waiting.\n    *   It saves money because you don\'t pay for shared resources that might be idle.\n\n4.  **Refine for

## 4. Visual understanding (VLM) — Qwen

`video.understand()` samples frames and asks the sandbox-backed VLM to describe them. Routing is via `sandbox_id` + `model` **inside the analyzer config**. This is the public sandbox path for image understanding today.

In [17]:
visual_understanding = video.understand(
    segmentation={"type": "time", "seconds": 8},
    sampling={"strategy": "uniform", "frame_count": 3},
    analyzers=[
        {
            "type": "vlm",
            "name": "scene_description",
            "config": {
                "model": TEXT_AND_VISION_MODEL,
                "sandbox_id": sandbox.id,
                "prompt": (
                    "Describe the people, objects, setting, and actions visible "
                    "in these images. Be concise and factual."
                ),
                "temperature": 0.1,
                "max_output_tokens": 300,
            },
        }
    ],
)

visual_understanding.wait_until_complete(timeout=1800, poll_interval=10)
if not visual_understanding.is_successful:
    raise RuntimeError(
        f"Visual understanding failed with status {visual_understanding.status}"
    )

visual_output = visual_understanding.get_analyzer("scene_description").get_output()
print("Understanding ID:", visual_understanding.id)
visual_output

Understanding ID: und_756b00fa991d464f


{'metadata': {},
 'name': 'scene_description',
 'scenes': [{'data': {'text': 'There are no people, objects, setting, or actions visible in these images. The images display a single, repeated photograph of a gold-colored speaker grille, likely from a high-fidelity audio system. The grille features a series of concentric, angular cutouts designed to allow sound to pass through while protecting the internal components. The background is a uniform, light beige or off-white color, providing a neutral backdrop that emphasizes the metallic sheen and geometric pattern of the grille. There is no indication of movement, context, or activity within the frame.'},
   'end': None,
   'metadata': {},
   'scene_id': 'scene_000000',
   'start': None},
  {'data': {'text': 'There are no people, objects, setting, or actions visible in the provided image. The image displays a single, repeated graphic element: a gold-colored, geometric pattern resembling a stylized "X" or cross, set against a white backgrou

## 5. Object detection — RT-DETR

Same `video.understand()` surface with an `object_detection` analyzer. Output is normalized detections (labels, counts, optional bounding boxes).

In [18]:
detection_understanding = video.understand(
    segmentation={"type": "time", "seconds": 8},
    sampling={"strategy": "uniform", "frame_count": 3},
    analyzers=[
        {
            "type": "object_detection",
            "name": "objects",
            "config": {
                "model": OBJECT_DETECTION_MODEL,
                "sandbox_id": sandbox.id,
                "confidence_threshold": 0.35,
                "include_bounding_boxes": True,
            },
        }
    ],
)

detection_understanding.wait_until_complete(timeout=1800, poll_interval=10)
if not detection_understanding.is_successful:
    raise RuntimeError(
        f"Object detection failed with status {detection_understanding.status}"
    )

object_output = detection_understanding.get_analyzer("objects").get_output()
print("Understanding ID:", detection_understanding.id)
object_output

Understanding ID: und_1f4fc125b8ed47f1


{'metadata': {},
 'name': 'objects',
 'scenes': [{'data': {'frames': [{'detections': [{'box': [0.1024,
         0.1482,
         0.9438,
         0.9976],
        'box_format': 'xyxy_normalized',
        'label': 'person',
        'score': 0.9213},
       {'box': [0.6643, 0.0838, 0.9992, 0.5668],
        'box_format': 'xyxy_normalized',
        'label': 'elephant',
        'score': 0.8713},
       {'box': [0.4512, 0.1106, 0.7457, 0.5159],
        'box_format': 'xyxy_normalized',
        'label': 'elephant',
        'score': 0.5105}],
      'frame_id': 'frame-scene-0.000-8.000-2.000',
      'timestamp_sec': 2.0},
     {'detections': [{'box': [0.6586, 0.1268, 1.0, 0.5978],
        'box_format': 'xyxy_normalized',
        'label': 'elephant',
        'score': 0.8801},
       {'box': [0.1575, 0.2595, 0.8746, 0.9975],
        'box_format': 'xyxy_normalized',
        'label': 'person',
        'score': 0.8629},
       {'box': [0.5979, 0.6386, 0.8587, 0.9993],
        'box_format': 'xyxy_norm

## 6. Text-to-speech — OmniVoice

Generation APIs are **asynchronous**: `generate_voice` returns a job; `wait()` returns a normal VideoDB `Audio` asset. Use `config={"instructions": ...}` to steer the voice (voice design).

In [19]:
from IPython.display import Audio as NotebookAudio, display

voice_job = collection.generate_voice(
    text="Welcome to VideoDB Sandbox Compute. This voice was generated on dedicated inference compute.",
    model_name=TTS_MODEL,
    sandbox_id=sandbox.id,
    config={"instructions": "A clear, friendly product-demo narrator"},
)

generated_audio = voice_job.wait(timeout=900, interval=5)
print("Job ID:", voice_job.id, "| Audio ID:", generated_audio.id)
display(NotebookAudio(url=generated_audio.generate_url()))

Job ID: 905f66c5-11e1-4333-b8fd-51e70aab825e | Audio ID: a-z-019f979b-c74f-7352-86f3-55d12432c215


## 7. Image generation — FLUX

Also async. FLUX runs on the `medium` tier; `config` is forwarded to the model (`size`, `num_inference_steps`, `guidance_scale`).

In [20]:
from IPython.display import Image as NotebookImage

image_job = collection.generate_image(
    prompt=(
        "A compact GPU server in a clean futuristic studio, cinematic lighting, "
        "high detail, no text"
    ),
    model_name=IMAGE_GENERATION_MODEL,
    sandbox_id=sandbox.id,
    config={
        "size": "1280x720",
        "num_inference_steps": 28,
        "guidance_scale": 4.0,
    },
)

generated_image = image_job.wait(timeout=900, interval=5)
print("Job ID:", image_job.id, "| Image ID:", generated_image.id)
display(NotebookImage(url=generated_image.generate_url()))

Job ID: 48c98104-5b34-4e7e-a291-f63e09859ce0 | Image ID: img-z-019f979c-1011-78d3-8dd6-e7019f6a76c4


## Model coverage

| Use case | Model | Min tier | This notebook | Deep-dive |
|---|---|---:|---|---|
| Text | `Qwen/Qwen3.5-9B` | small | ✅ §3 | — |
| Visual understanding | `Qwen/Qwen3.5-9B` | small | ✅ §4 | — |
| Object detection | `rtdetr-v2-r50vd` | small | ✅ §5 | — |
| Text-to-speech | `k2-fsa/OmniVoice` | small | ✅ §6 | `omnivoice_quickstart.ipynb` |
| Image generation | `black-forest-labs/FLUX.1-dev` | medium | ✅ §7 | `flux_quickstart.ipynb` |
| Speech-to-text | `openai/whisper-large-v3-turbo` | small | — | `whisper_quickstart.ipynb` (WAV input) |
| Audio generation | `stabilityai/stable-audio-open-1.0` | small | — | Not yet available via the SDK |

Single-model deep-dive notebooks live alongside this one in `guides/sandbox/`.

## 🚨 Cleanup — stop the sandbox

Billing is runtime-based, so stop the sandbox when you're done. `grace=True` lets accepted work finish before compute is released.

In [21]:
sandbox = conn.get_sandbox(sandbox.id)

if sandbox.status not in ("stopped", "failed"):
    sandbox.stop(grace=True)
    sandbox.wait_for_stop(timeout=300, interval=5)

print("Sandbox ID:", sandbox.id)
print("Final status:", sandbox.status)

Sandbox ID: bx-13832c0a9e4f4fa9
Final status: stopped
